#Import libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

print("Current working directory:", Path.cwd())

Current working directory: c:\Users\User\Desktop\Customer_Churn_Prediction\notebooks


#Locate the cleaned dataset

In [2]:
cwd = Path.cwd()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

PROJECT_ROOT = None

for candidate in candidate_roots:
    expected_file = (
        candidate
        / "data"
        / "processed"
        / "customer_churn_cleaned.csv"
    )

    if expected_file.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find customer_churn_cleaned.csv. "
        "Run 01_data_quality.ipynb first."
    )

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_churn_cleaned.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "models"
)

METRIC_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset:", DATA_PATH.resolve())
print("Dataset exists:", DATA_PATH.exists())

Project root: C:\Users\User\Desktop\Customer_Churn_Prediction
Dataset: C:\Users\User\Desktop\Customer_Churn_Prediction\data\processed\customer_churn_cleaned.csv
Dataset exists: True


#Load cleaned dataset

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Dataset loaded successfully.
Rows: 7043
Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
#Check the data before preprocessing

In [4]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nTarget values:")
print(df["Churn"].value_counts())

print("\nData types:")
print(df.dtypes)

Missing values: 0
Duplicate rows: 0

Target values:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


#Remove accidental ChurnFlag if present

In [6]:
if "ChurnFlag" in df.columns:
    df = df.drop(
        columns=["ChurnFlag"]
    )

print(df.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


#Separate customer ID

In [7]:
customer_ids = df["customerID"].copy()

customer_ids.head()

0    7590-VHVEG
1    5575-GNVDE
2    3668-QPYBK
3    7795-CFOCW
4    9237-HQITU
Name: customerID, dtype: str

#Create features X and target y

In [8]:
X = df.drop(
    columns=[
        "customerID",
        "Churn"
    ]
).copy()

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (7043, 19)
y shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


#Target percentages

In [9]:
target_distribution = (
    y.value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

target_distribution.index = [
    "No Churn",
    "Churn"
]

target_distribution

No Churn    73.46
Churn       26.54
Name: proportion, dtype: float64

#Define numerical features

In [10]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

numeric_features

['tenure', 'MonthlyCharges', 'TotalCharges']

#Define categorical features

In [11]:
categorical_features = [
    column
    for column in X.columns
    if column not in numeric_features
]

print(
    "Number of categorical features:",
    len(categorical_features)
)

categorical_features

Number of categorical features: 16


['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

#Verify feature groups

In [12]:
print("Numerical features:")
for column in numeric_features:
    print("-", column)

print("\nCategorical features:")
for column in categorical_features:
    print("-", column)

Numerical features:
- tenure
- MonthlyCharges
- TotalCharges

Categorical features:
- gender
- SeniorCitizen
- Partner
- Dependents
- PhoneService
- MultipleLines
- InternetService
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
- Contract
- PaperlessBilling
- PaymentMethod


#Train/test split

In [13]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    id_train,
    id_test
) = train_test_split(
    X,
    y,
    customer_ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


In [14]:
print("Training set churn distribution:")

print(
    y_train
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

print("\nTesting set churn distribution:")

print(
    y_test
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

Training set churn distribution:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Testing set churn distribution:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


#Create numerical transformer

In [15]:
numeric_transformer = (
    StandardScaler()
)

numeric_transformer

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


#Create categorical transformer

In [16]:
categorical_transformer = (
    OneHotEncoder(
        handle_unknown="ignore"
    )
)

categorical_transformer

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_categ

#Build preprocessing pipeline

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [ ]:
#Fit ONLY on training data

In [18]:
preprocessor.fit(
    X_train
)

print(
    "Preprocessor fitted successfully "
    "using training data only."
)

Preprocessor fitted successfully using training data only.


#Transform training and testing data

In [19]:
X_train_transformed = (
    preprocessor.transform(
        X_train
    )
)

X_test_transformed = (
    preprocessor.transform(
        X_test
    )
)

print(
    "Original X_train shape:",
    X_train.shape
)

print(
    "Transformed X_train shape:",
    X_train_transformed.shape
)

print(
    "Original X_test shape:",
    X_test.shape
)

print(
    "Transformed X_test shape:",
    X_test_transformed.shape
)

Original X_train shape: (5634, 19)
Transformed X_train shape: (5634, 46)
Original X_test shape: (1409, 19)
Transformed X_test shape: (1409, 46)


In [20]:
#Get transformed feature names
feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Total transformed features:",
    len(feature_names)
)

feature_names

Total transformed features: 46


array(['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges',
       'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_0',
       'cat__SeniorCitizen_1', 'cat__Partner_No', 'cat__Partner_Yes',
       'cat__Dependents_No', 'cat__Dependents_Yes',
       'cat__PhoneService_No', 'cat__PhoneService_Yes',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__TechSupport_No',
       'cat__TechSupport_No internet service', 'cat__TechSupport_Yes',
       'cat__StreamingTV_No', 'cat__StreamingTV_No

#Check transformed sample

In [21]:
print(
    X_train_transformed[:3]
)

[[ 0.10237124 -0.52197565 -0.2622572   0.          1.          1.
   0.          1.          0.          1.          0.          1.
   0.          0.          1.          0.          1.          0.
   0.          1.          0.          0.          1.          0.
   0.          0.          0.          1.          1.          0.
   0.          0.          0.          1.          0.          0.
   1.          1.          0.          0.          1.          0.
   0.          0.          1.          0.        ]
 [-0.71174346  0.33747781 -0.50363479  0.          1.          1.
   0.          0.          1.          0.          1.          0.
   1.          1.          0.          0.          0.          1.
   0.          0.          0.          1.          1.          0.
   0.          1.          0.          0.          1.          0.
   0.          1.          0.          0.          1.          0.
   0.          1.          0.          0.          1.          0.
   0.          0.        

#Save raw training features

In [22]:
X_train.to_csv(
    PROCESSED_DIR
    / "X_train.csv",
    index=False
)

X_test.to_csv(
    PROCESSED_DIR
    / "X_test.csv",
    index=False
)

print(
    "X_train and X_test saved."
)

X_train and X_test saved.


#Save Targets

In [23]:
y_train_reset = (
    y_train
    .reset_index(drop=True)
)

y_test_reset = (
    y_test
    .reset_index(drop=True)
)

y_train_reset.to_csv(
    PROCESSED_DIR
    / "y_train.csv",
    index=False,
    header=["Churn"]
)

y_test_reset.to_csv(
    PROCESSED_DIR
    / "y_test.csv",
    index=False,
    header=["Churn"]
)

print(
    "y_train and y_test saved."
)

y_train and y_test saved.


#Save test customer IDs

In [24]:
id_train_reset = (
    id_train
    .reset_index(drop=True)
)

id_test_reset = (
    id_test
    .reset_index(drop=True)
)

id_train_reset.to_csv(
    PROCESSED_DIR
    / "train_customer_ids.csv",
    index=False,
    header=["customerID"]
)

id_test_reset.to_csv(
    PROCESSED_DIR
    / "test_customer_ids.csv",
    index=False,
    header=["customerID"]
)

print(
    "Customer IDs saved."
)

Customer IDs saved.


#Save preprocessing object

In [25]:
PREPROCESSOR_PATH = (
    MODEL_DIR
    / "preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print(
    "Preprocessor saved to:"
)

print(
    PREPROCESSOR_PATH
)

Preprocessor saved to:
c:\Users\User\Desktop\Customer_Churn_Prediction\outputs\models\preprocessor.joblib


#Check

In [26]:
print(
    "Preprocessor exists:",
    PREPROCESSOR_PATH.exists()
)

Preprocessor exists: True


#Create processed dataset for AWS/Athena

In [27]:
processed_df = df.copy()

processed_df["ChurnFlag"] = (
    processed_df["Churn"]
    .map({
        "No": 0,
        "Yes": 1
    })
)

processed_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [ ]:
#Save processed dataset

In [28]:
ATHENA_DATA_PATH = (
    PROCESSED_DIR
    / "customer_churn_processed.csv"
)

processed_df.to_csv(
    ATHENA_DATA_PATH,
    index=False
)

print(
    "Processed analytical dataset saved:"
)

print(
    ATHENA_DATA_PATH
)

Processed analytical dataset saved:
c:\Users\User\Desktop\Customer_Churn_Prediction\data\processed\customer_churn_processed.csv


In [ ]:
#Create preprocessing summary

In [29]:
preprocessing_summary = pd.DataFrame({
    "Item": [
        "Total records",
        "Training records",
        "Testing records",
        "Original model features",
        "Numerical features",
        "Categorical features",
        "Transformed features",
        "Training churn rate (%)",
        "Testing churn rate (%)"
    ],

    "Value": [
        len(df),
        len(X_train),
        len(X_test),
        X.shape[1],
        len(numeric_features),
        len(categorical_features),
        len(feature_names),
        round(
            y_train.mean() * 100,
            2
        ),
        round(
            y_test.mean() * 100,
            2
        )
    ]
})

preprocessing_summary

,Item,Value
0,Total records,7043.00
1,Training records,5634.00
2,Testing records,1409.00
3,Original model features,19.00
4,Numerical features,3.00
5,Categorical features,16.00
6,Transformed features,46.00
7,Training churn rate (%),26.54
8,Testing churn rate (%),26.54


#Save preprocessing summary

In [30]:
preprocessing_summary.to_csv(
    METRIC_DIR
    / "preprocessing_summary.csv",
    index=False
)

print(
    "Preprocessing summary saved."
)

Preprocessing summary saved.


#Final verification

In [31]:
files_to_check = [
    PROCESSED_DIR / "X_train.csv",
    PROCESSED_DIR / "X_test.csv",
    PROCESSED_DIR / "y_train.csv",
    PROCESSED_DIR / "y_test.csv",
    PROCESSED_DIR / "train_customer_ids.csv",
    PROCESSED_DIR / "test_customer_ids.csv",
    PROCESSED_DIR / "customer_churn_processed.csv",
    MODEL_DIR / "preprocessor.joblib",
    METRIC_DIR / "preprocessing_summary.csv"
]

for file in files_to_check:
    print(
        file.name,
        "→",
        file.exists()
    )

X_train.csv → True
X_test.csv → True
y_train.csv → True
y_test.csv → True
train_customer_ids.csv → True
test_customer_ids.csv → True
customer_churn_processed.csv → True
preprocessor.joblib → True
preprocessing_summary.csv → True
